# 01 — EDA de retornos y quiebres estructurales (#5, #6)

Requiere haber corrido `python -m src.data.download_data --with-exogenous` y `python -m src.data.clean_data` (o `python run_pipeline.py --steps data`).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
import os; os.chdir(ROOT)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.width", 160)
%matplotlib inline

In [ ]:
from src import config
from src.utils.io import load_frame
processed = {n: load_frame(config.PROCESSED_DIR / f'{n}.csv') for n in config.ASSETS}
{n: (df.index.min().date(), df.index.max().date(), len(df)) for n, df in processed.items()}

## Tabla 1 — estadísticos descriptivos y pruebas
`***` p<0.01, `**` p<0.05, `*` p<0.10. Q = Ljung-Box, ARCH-LM = Engle.

In [ ]:
from src.analysis.eda import descriptive_table, format_table
tab = descriptive_table({n: df['log_return'] for n, df in processed.items()})
format_table(tab)

**Cómo leerla:** curtosis en exceso > 0 y JB/Shapiro significativos → colas pesadas (no normalidad). Q²(10) y ARCH-LM significativos → *volatility clustering*, lo que justifica modelos GARCH.

In [ ]:
from src.analysis.eda import plot_price_returns, plot_distribution, plot_acf_returns
for n, df in processed.items():
    plot_price_returns(df['price'], df['log_return'], n.upper()); plt.show()
    plot_distribution(df['log_return'], n.upper()); plt.show()
    plot_acf_returns(df['log_return'], title=n.upper()); plt.show()

## Quiebres estructurales en varianza (ICSS)
Se usa el estadístico κ2 de Sansó et al. (2004), robusto a colas pesadas y GARCH. El IT original se muestra para comparar (suele sobredetectar).

In [ ]:
from src.features.regimes import break_dates, regime_table
from src.analysis.eda import plot_breaks
for n, df in processed.items():
    r = df['log_return']
    b = break_dates(r, method='kappa2')
    print(n, '→ κ2:', len(b), 'quiebres | IT:', len(break_dates(r, method='it')))
    display(regime_table(r, b))
    plot_breaks(r, b, f'{n.upper()} — ICSS κ2'); plt.show()

## Notas para el reporte
- Relacionar cada fecha de quiebre con eventos (p. ej. 2016 elecciones EE.UU., 2020 COVID-19, 2022 alzas de tasas).